In [2]:
import pandas as pd
import numpy as np

In [3]:
# graph preprocessing visualization functions
#!/usr/bin/env python3
"""
Convert a saved LiNGAM adjacency .npz into Gephi-ready nodes.csv and edges.csv.

Expected NPZ keys:
- adjacency: (p, p) numpy array, where adjacency[i, j] = effect of j -> i
- features:  (p,) array of feature names in the same order as adjacency indices
"""

from __future__ import annotations

import os
def npz_to_gephi(
    npz_path: str,
    outdir: str = "",
    nodes_filename: str = "nodes.csv",
    edges_filename: str = "edges.csv",
    weight_threshold: float | None = None,
    absolute_threshold: bool = True,
    keep_self_loops: bool = False,
) -> tuple[str, str]:
    """
    Load adjacency + feature names from NPZ and export Gephi nodes + edges CSVs.

    Parameters
    ----------
    npz_path : str
        Path to the saved .npz file (created via np.savez_compressed).
    outdir : str
        Output directory for Gephi CSV files.
    nodes_filename : str
        Name of nodes CSV file.
    edges_filename : str
        Name of edges CSV file.
    weight_threshold : float | None
        If set, keep only edges with |weight| >= threshold (default uses abs if absolute_threshold=True).
        Useful to reduce graph size for visualization.
    absolute_threshold : bool
        If True, threshold is applied on abs(weight); else on weight directly.
    keep_self_loops : bool
        If True, keep i->i edges (usually False).

    Returns
    -------
    (nodes_path, edges_path) : tuple[str, str]
        Paths to the exported nodes and edges CSV files.
    """
    if not os.path.isfile(npz_path):
        raise FileNotFoundError(f"NPZ file not found: {npz_path}")

    os.makedirs(outdir, exist_ok=True)

    data = np.load(npz_path, allow_pickle=True)
    if "adjacency" not in data or "features" not in data:
        raise ValueError("NPZ must contain keys: 'adjacency' and 'features'.")

    A = data["adjacency"]
    features = data["features"].tolist()

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f"Adjacency must be square; got shape {A.shape}.")
    if len(features) != A.shape[0]:
        raise ValueError(
            f"features length ({len(features)}) must match adjacency size ({A.shape[0]})."
        )

    p = A.shape[0]

    # -----------------------------
    # Nodes file (Gephi)
    # -----------------------------
    # Gephi nodes CSV minimal columns: Id, Label
    # We'll use feature name as both Id and Label to keep things simple.
    nodes_df = pd.DataFrame({"Id": features, "Label": features})
    nodes_path = os.path.join(outdir, nodes_filename)
    nodes_df.to_csv(nodes_path, index=False)

    # -----------------------------
    # Edges file (Gephi)
    # -----------------------------
    # LiNGAM convention: A[i, j] = effect of j -> i
    # So edge Source = features[j], Target = features[i]
    rows, cols = np.where(A != 0)

    if not keep_self_loops:
        mask = rows != cols
        rows, cols = rows[mask], cols[mask]

    weights = A[rows, cols]

    if weight_threshold is not None:
        if absolute_threshold:
            keep = np.abs(weights) >= weight_threshold
        else:
            keep = weights >= weight_threshold
        rows, cols, weights = rows[keep], cols[keep], weights[keep]

    edges_df = pd.DataFrame(
        {
            "Source": [features[j] for j in cols],
            "Target": [features[i] for i in rows],
            "Weight": weights,
        }
    )

    # Optional: add AbsWeight for sorting/debugging
    edges_df["AbsWeight"] = np.abs(edges_df["Weight"])
    edges_df = edges_df.sort_values("AbsWeight", ascending=False)

    edges_path = os.path.join(outdir, edges_filename)
    edges_df.to_csv(edges_path, index=False)

    return nodes_path, edges_path





In [34]:
#data = np.load("./real_data_application/exp2/output/roots_screening/adjacency_model_run_time.npz", allow_pickle=True)
data = np.load("./real_data_application/exp4/output/lingam/adjacency_model_run_time_lingam_30_percent.npz", allow_pickle=True)
A = data["adjacency"]
features = data["features"].tolist()
adj_df = pd.DataFrame(A, index=features, columns=features)

In [36]:
nodes_path, edges_path = npz_to_gephi(
                                      npz_path="./real_data_application/exp4/output/lingam/adjacency_model_run_time_lingam_30_percent.npz",
                                      outdir="./real_data_application/exp4/output/lingam/",
                                      # Set a threshold if you want fewer edges in Gephi, e.g. 0.05 or 0.1
                                      weight_threshold=None,
                                      absolute_threshold=True,
                                      keep_self_loops=False,
                                      )